In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. KHỞI TẠO VÀ ĐỌC DỮ LIỆU 

In [3]:
file_name = 'Customer_cancellation.csv'
df = pd.read_csv(file_name)
print(f"Đã đọc file: {file_name}")
print(f"Số lượng dòng ban đầu: {len(df)}")

Đã đọc file: Customer_cancellation.csv
Số lượng dòng ban đầu: 7045


## 2. LÀM SẠCH VÀ CHUẨN HÓA DỮ LIỆU CƠ BẢN

In [5]:
df[df['Customer ID'].isna()]

,Customer ID,Gender,Age,Married,Number of Dependents,City,Number of Referrals,Tenure in Months,Offer,Phone Service,...,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason,Churned,Stayed,Unnamed: 36,Unnamed: 37,Unnamed: 38
7043,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.dropna(axis=0, how='all', inplace=True)
df.dropna(axis=1, how='all', inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 0 to 7042
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   float64
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   float64
 5   City                               7043 non-null   object 
 6   Number of Referrals                7043 non-null   float64
 7   Tenure in Months                   7043 non-null   float64
 8   Offer                              3166 non-null   object 
 9   Phone Service                      7043 non-null   object 
 10  Avg Monthly Long Distance Charges  6361 non-null   float64
 11  Multiple Lines                     6361 non-null   object 
 1

In [9]:
#Kiểm tra giá trị null trong Total Charges
print(f"giá trị NaN trong Total Charges: {df['Total Charges'].isna().sum()}")

giá trị NaN trong Total Charges: 0


In [12]:
# B. Chuẩn hóa Cột Dịch vụ
# Thay thế 'No internet service' / 'No phone service' bằng 'No' để dễ tính toán
internet_addons = [
    'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan',
    'Premium Tech Support', 'Streaming TV', 'Streaming Movies',
    'Streaming Music', 'Unlimited Data'
]
for col in internet_addons:
    df[col] = df[col].replace('No internet service', 'No')

df['Multiple Lines'] = df['Multiple Lines'].replace('No phone service', 'No')

In [13]:
# C. Tạo Cột Churn Flag
df['Churn Flag'] = np.where(df['Customer Status'] == 'Churned', 1, 0)

In [14]:
print(df['Churn Flag'])

0       0
1       1
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    0
7042    0
Name: Churn Flag, Length: 7043, dtype: int64


## 3. TÍNH TOÁN CỘT MỚI (FEATURE ENGINEERING)

In [20]:
# A. Tính toán cho Vòng đời Khách hàng (Dashboard 3 & 7)
# 1. Tenure Group
bins = [0, 3, 6, 12, 24, 72]
labels = ['0-3M', '4-6M', '7-12M', '13-24M', '24M+']
df['Tenure Group'] = pd.cut(df['Tenure in Months'], bins=bins, labels=labels, right=True)

# 2. Early Churn Flag (Churn trong <= 6 tháng)
df['Early Churn Flag'] = np.where(
    (df['Tenure in Months'] <= 6) & (df['Churn Flag'] == 1),
    1,
    0
)

In [21]:
print(df['Tenure Group'].unique())
print(df['Early Churn Flag'])

['0-3M', '4-6M', '7-12M', '13-24M', '24M+']
Categories (5, object): ['0-3M' < '4-6M' < '7-12M' < '13-24M' < '24M+']
0       0
1       1
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    0
7042    0
Name: Early Churn Flag, Length: 7043, dtype: int64


In [22]:
# B. Tính toán cho Gắn kết Dịch vụ (Dashboard 4 & 7)
addon_cols = [
    'Online Security', 'Online Backup', 'Device Protection Plan',
    'Premium Tech Support', 'Streaming TV', 'Streaming Movies',
    'Streaming Music', 'Unlimited Data'
]
# Đếm số lượng Add-ons đang sử dụng (Yes = 1, No = 0)
df_addons = df[addon_cols].apply(lambda x: np.where(x == 'Yes', 1, 0))
df['Num Internet Add-ons'] = df_addons.sum(axis=1)

In [23]:
print(df['Num Internet Add-ons'])

0       1
1       1
2       2
3       1
4       0
       ..
7038    7
7039    6
7040    8
7041    4
7042    7
Name: Num Internet Add-ons, Length: 7043, dtype: int64


In [24]:
# C. Tính toán cho Tài chính (Dashboard 5 & 7)
# 1. Total Refund Rate (Tránh chia cho 0)
df['Total Refund Rate'] = np.where(
    df['Total Charges'] > 0,
    df['Total Refunds'] / df['Total Charges'],
    0
)

In [25]:
# 2. High Value Customer Flag (Dựa trên Total Revenue Q75)
q75_revenue = df['Total Revenue'].quantile(0.75)
df['High Value Flag'] = np.where(
    df['Total Revenue'] >= q75_revenue,
    'High Value',
    'Standard'
)

## 4. PHÂN KHÚC HÀNH ĐỘNG (SEGMENTATION) - DASHBOARD 7

In [26]:
# Tính Median Monthly Charge để dùng làm ngưỡng
median_monthly_charge = df['Monthly Charge'].median()

In [27]:
# Định nghĩa các điều kiện phân khúc
conditions = [
    # 1. Loyal Customer (Ưu tiên)
    (df['Tenure in Months'] > 24) &
    (df['Contract'].isin(['One Year', 'Two Year'])) &
    (df['High Value Flag'] == 'High Value'),

    # 2. High Risk (Rủi ro Churn sớm + Charge cao + Hợp đồng linh hoạt + Không bảo vệ)
    (df['Tenure in Months'] <= 6) &
    (df['Monthly Charge'] >= median_monthly_charge) &
    (df['Premium Tech Support'] == 'No') &
    (df['Contract'] == 'Month-to-Month'),

    # 3. Price Sensitive (Hoàn tiền cao hoặc Churn vì giá/đối thủ)
    (df['Total Refund Rate'] > 0.05) |
    (df['Churn Category'].isin(['Competitor', 'Price'])),
    
    # 4. Low Engagement (Ít Add-ons và loại Internet chậm)
    (df['Num Internet Add-ons'] <= 1) &
    (df['Internet Type'] == 'DSL')

]

In [29]:
# Định nghĩa các tên Segment
choices = [
    'Loyal Customer (Upsell)',
    'High Risk (Retention Offer)',
    'Price Sensitive (Price Review)',
    'Low Engagement (Bundle Offer)'
]

# Áp dụng np.select để tạo cột Customer Segment
df['Customer Segment'] = np.select(conditions, choices, default='Standard/Other')

# In kết quả kiểm tra
print("\n--- Phân phối Customer Segment ---")
print(df['Customer Segment'].value_counts(normalize=True).round(3) * 100)


--- Phân phối Customer Segment ---
Customer Segment
Standard/Other                    59.6
Loyal Customer (Upsell)           20.0
Price Sensitive (Price Review)    11.5
High Risk (Retention Offer)        6.6
Low Engagement (Bundle Offer)      2.4
Name: proportion, dtype: float64


In [30]:
output_file = 'processed_customer_churn_data.csv'
df.to_csv(output_file, index=False)

print(f"\n✅ Dữ liệu đã được xử lý và lưu tại: {output_file}")


✅ Dữ liệu đã được xử lý và lưu tại: processed_customer_churn_data.csv
